In [1]:
import os
import torch

from tqdm import tqdm

from stock_gpt import StockGPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockGPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [ ]:
def batch_loss(x, y, model):
    """
    Assume x and y are on the correct device already \n
    Returns NLL for back propagation
    """
    mean, std = model(x)
    y_norm = (y - model.target_mean) / model.target_std
    dist = torch.distributions.Normal(mean, std)
    return -dist.log_prob(y_norm).mean()

def eval_loss(data_loader, model, device, max_batches = float("inf"), pbar = None, desc="") -> dict: 
    """
    Returns a dict of MAE loss and Negative Log Loss 
    """
    num_batches = min(len(data_loader), max_batches)
    avg_mae = 0
    avg_pmae = 0
    avg_nll = 0
    avg_std = 0

    for i, (p, t) in enumerate(data_loader):
        if i == num_batches:
            break

        p = p.to(device, non_blocking=True)
        t = t.to(device, non_blocking=True)

        #* MAE
        mean, std = model(p)
        t_norm = (t - model.target_mean) / model.target_std
        mae = torch.abs(mean - t_norm)
        avg_mae += (mae - avg_mae) / (i+1)

        #* MAE %
        t_real = mean * model.target_std + model.target_mean
        pmae = torch.abs(t_real - t) / torch.abs(t).clamp_min(1e-8) * 100
        avg_pmae += (pmae - avg_pmae) / (i+1)

        #*NLL
        dist = torch.distributions.Normal(mean, std)
        nll = -dist.log_prob(t_norm)
        avg_nll += (nll - avg_nll) / (i+1)

        #*STD
        col_std = std.mean(dim=(0, 1))
        avg_std += (col_std - avg_std) / (i+1)

        if pbar is not None:
            pbar.update(1)
            if i % max(1,int(num_batches*0.001))==0:
                pbar.set_description(f"{desc} ({i}/{num_batches}) [{pbar.n}/{pbar.total}]")
    return {"NLL": avg_nll, "STD": avg_std, "MAE": avg_mae, "PMAE": avg_pmae}

## MODEL TRAINING ---------------------------

In [ ]:
def load_model(path, model, device, optimizer=None, cuda_scaler=None):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model"])
    if optimizer is not None and "optimizer" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if cuda_scaler is not None and "cuda_scaler" in checkpoint:
        cuda_scaler.load_state_dict(checkpoint["cuda_scaler"])
    return checkpoint

def evaluate_model(train_dl, val_dl, model, device, eval_bs, pbar = None):
    """
    Returns a list of dictionaries, with each dictionary correspoding to a function in eval_fns
    """
    with torch.no_grad():
        train_metrics = eval_loss(train_dl, model, device, eval_bs, pbar,
                                    desc="Evaluating model on training data...")
        val_metrics = eval_loss(val_dl, model, device, eval_bs, pbar,
                                  desc="Evaluating model on validation data...")    
    return train_metrics, val_metrics

def evaluate_best_model(model, device, optimizer, cuda_scaler, train_dl, val_dl,
                        eval_bs, pbar = None, reevaluate = False):
    if os.path.exists(model.best_path):
        checkpoint = load_model(model.best_path, model, device, optimizer, cuda_scaler)
        if reevaluate is False:
            return checkpoint["train_losses"][-1], checkpoint["val_losses"][-1]
        else:
            return evaluate_model(train_dl, val_dl, model, device, eval_bs, pbar)
    else:
        raise FileNotFoundError("Best parameters of the model could not be found")

def train_model_cuda(model, device, optimizer, cuda_scaler, max_epochs,
                     train_dl, val_dl, eval_bs):
    #* LOADS MODEL
    if os.path.exists(model.checkpoint_path):
        print("Continuing from previous checkpoint...")
        checkpoint = load_model(model.checkpoint_path, model, device, optimizer, cuda_scaler)
        bvm, epoch, train_losses, val_losses = (
            checkpoint["bvm"], checkpoint["epoch"]+1, checkpoint["train_losses"], checkpoint["val_losses"]
        )
    elif os.path.exists(model.best_path):
        print("Continuing from best parameter state...")
        checkpoint = load_model(model.best_path, model, device, optimizer, cuda_scaler)
        bvm, epoch, train_losses, val_losses = (
            checkpoint["bvm"], checkpoint["epoch"]-2, checkpoint["train_losses"], checkpoint["val_losses"]
        ) #* Does at most an additional 3 checkpoints when checkpoint path file is deleted and best exists
    else:
        bvm, epoch, train_losses, val_losses = float("inf"), 0, [], []

    eval_steps = min(eval_bs, len(train_dl)) + min(eval_bs, len(val_dl))
    pbar = tqdm(total=(max_epochs-epoch)*(len(train_dl)+eval_steps), desc=f"Setting up...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False, delay=0.5)
    pbar.write((f"Epoch {epoch+1}:\n"))
    try:
        for epoch in range(epoch, max_epochs):
            #* TRAINS MODEL
            model.train()
            for x, y in train_dl:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    loss = batch_loss(x, y, model).mean()
                cuda_scaler.scale(loss).backward()
                cuda_scaler.step(optimizer)
                cuda_scaler.update()

                pbar.update(1)
                if (pbar.n % max(1,int(pbar.total*0.001))==0):
                    pbar.set_description(f"Training the {model.cfg['name']}... [{pbar.n}/{pbar.total}]")

            #* EVALUATES MODEL
            model.eval()
            pbar.set_description(f"Evaluating Epoch {epoch}... [{pbar.n}/{pbar.total}]")
            train_metrics, val_metrics = evaluate_model(train_dl, val_dl, model, device, eval_bs, pbar)
            pbar.write((
                        f"Epoch {epoch+1}:\n"
                        f"Training Loss:\n"
                        f"   (MAE) {train_metrics['MAE'].mean()}\n"
                        f"   (NLL) {train_metrics['NLL'].mean()}\n"
                        f"Validation Loss:\n"
                        f"   (MAE) {val_metrics['MAE'].mean()}\n"
                        f"   (NLL) {val_metrics['NLL'].mean()}\n"))
            train_losses.append(train_metrics)
            val_losses.append(val_metrics)

            #* SAVES MODEL
            cvm = val_metrics['NLL'].mean().item()
            checkpoint = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "cuda_scaler": cuda_scaler.state_dict(),
                "epoch": epoch,
                "train_losses": train_losses,
                "val_losses": val_losses,
                "bvm": bvm
            }
            if (cvm < bvm):
                bvm = cvm
                checkpoint["bvm"] = cvm
                torch.save(checkpoint, model.best_path)
            pbar.write((f"Best Validation: {bvm}\n{'-'*100}\n"))
            torch.save(checkpoint, model.checkpoint_path)
    finally:
            pbar.close()
    print("Finished")
    return train_losses, val_losses

In [5]:
torch.manual_seed(1234)
dls, train_norms = build_dataloaders(path_data_preprocessor)

Building DataLoaders...


In [9]:
def model_setup(model_cls, cfg, train_norms, device, optimizer_cls, lr, weight_decay, scaler_cls, scale_type):
    model = model_cls(cfg, train_norms)
    model.to(device)
    model_params = sum(p.numel() for p in model.parameters())
    print(model_params)
    optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = scaler_cls(scale_type)
    return model, model_params, optimizer, scaler

optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 10
eval_bs = 1000

stockGPT, stockGPT_params, o1, s1 = model_setup(StockGPT, StockGPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, o2, s2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

model_train_losses, model_val_losses = train_model_cuda(stockGPT, device, o1, s1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, o2, s2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


Input Norm: torch.Size([13])|torch.Size([13])
Target Norm: torch.Size([4])|torch.Size([4])
3181824
5632
Continuing from previous checkpoint...


Epoch 16:

Finished


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 1:



|█         | 10.0% (01:07) Evaluating model on validation data... (612/613) [5304/53040]:                 

...

|████████  | 80.0% (07:49) Evaluating model on validation data... (612/613) [42432/53040]: 

Epoch 8:
Training Loss:
   (MAE) 0.0703330934047699
   (NLL) -0.04163583368062973
Validation Loss:
   (MAE) 0.07391481101512909
   (NLL) -0.07505202293395996

Best Validation: -2.4611973762512207
----------------------------------------------------------------------------------------------------



|█████████ | 90.0% (08:44) Evaluating model on validation data... (612/613) [47736/53040]: 

Epoch 9:
Training Loss:
   (MAE) 0.05919178947806358
   (NLL) -0.3386261463165283
Validation Loss:
   (MAE) 0.06387116760015488
   (NLL) -0.1416182816028595

Best Validation: -2.4611973762512207
----------------------------------------------------------------------------------------------------



Epoch 10:
Training Loss:
   (MAE) 0.022746684029698372
   (NLL) -0.766963005065918
Validation Loss:
   (MAE) 0.02410934306681156
   (NLL) -0.4705672860145569

Best Validation: -2.4611973762512207
----------------------------------------------------------------------------------------------------

Finished


## Model Analysis -------------------------

In [28]:
def process_losses(losses: list[dict], key = "MAE Loss"):
    if key == "STD":
        return [loss_dict[key] for loss_dict in losses]
    return [loss_dict[key].mean(dim=(0,1)) for loss_dict in losses]

def tensor_to_string(t, cs):
    return "".join(f"{v.item():<{cs}.4f}" for v in t)

def format_num(n):
    if n >= 1e9:
        return f"{n / 1e9:.1f}B"
    if n >= 1e6:
        return f"{n / 1e6:.1f}M"
    if n >= 1e3:
        return f"{n / 1e3:.1f}K"
    return str(n)

def print_loss_analysis(losses, model_names, parameters, col_names, key, cs = 9):
    title = f"{key}\n"
    bound = f"\n{'-'*110}\n\n"
    header = f"{' '*20}"+"".join(f"{col_name:<{cs}}" for col_name in col_names)+"\n"
    rows = "".join(
        f"{row_name}: {parameters[i]}\n"
        f"    Training:       {tensor_to_string(losses[i*3], cs)}  >  {losses[i*3].mean():.4f}\n"
        f"    Validation:     {tensor_to_string(losses[i*3+1], cs)}  >  {losses[i*3+1].mean():.4f}\n"
        f"    Testing:     {tensor_to_string(losses[i*3+2], cs)}  >  {losses[i*3+2].mean():.4f}\n"
        f"    "
        f"\n"
    for i, row_name in enumerate(model_names))
    output = [
        bound,
        title,
        bound,
        header,
        rows,
        bound
    ]
    print("".join(output))

def test_model(test_dl, model, device, eval_bs, pbar=None):
    """
    Evaluates model on testing data only.
    """
    model.eval()
    with torch.no_grad():
        test_metrics = eval_loss(test_dl, model, device, eval_bs, pbar, desc="Evaluating model on testing data...")
    return (test_metrics,)

In [27]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, o2, s2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
gpt_losses = evaluate_best_model(stockGPT, device, o1, s1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
gpt_test_losses = test_model(dls["test"], stockGPT, device, eval_bs, analysis_pbar)

for key, features in [("NLL", StockGPT_cfg["target_features"]), 
                      ("STD", [f"{feature}_std" for feature in StockGPT_cfg["target_features"]]), 
                      ("MAE", StockGPT_cfg["target_features"]), 
                      ("PMAE", StockGPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(gpt_losses + gpt_test_losses 
                                + linear_losses + linear_test_losses 
                                + naive_losses + naive_test_losses, key),
                ["StockGPT", "LinearModel", "NaiveModel"],
                [f"{format_num(stockGPT_params)}", f"{format_num(linearModel_params)}", f"0"],
                features, key)

|██████████| 100.0% (01:54) Evaluating model on testing data... (387/388) [6003/6003]:                    

NameError: name 'print_loss_analysis' is not defined

## Model Testing -------------------------

In [ ]:
test_steps = min(eval_bs, len(dls["test"]))
test_pbar = tqdm(total=3*test_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, test_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, test_pbar)
gpt_test_losses = test_model(dls["test"], stockGPT, device, eval_bs, test_pbar)

for key, features in [("NLL", StockGPT_cfg["target_features"]), 
                      ("STD", [f"{feature}_std" for feature in StockGPT_cfg["target_features"]]), 
                      ("MAE", StockGPT_cfg["target_features"]), 
                      ("PMAE", StockGPT_cfg["target_features"])]:
    print_test_losses(process_losses(gpt_test_losses + linear_test_losses + naive_test_losses, key),
                ["StockGPT", "LinearModel", "NaiveModel"],
                [f"{format_num(stockGPT_params)}", f"{format_num(linearModel_params)}", f"0"],
                features, key)

|██████████| 100.0% (00:21) Evaluating model on testing data... (387/388) [1164/1164]:                    


--------------------------------------------------------------------------------------------------------------

NLL

--------------------------------------------------------------------------------------------------------------

                    o        h        l        c        
StockGPT: 3.2M
    Testing:       -3.8986  -3.7634  -3.7847  -3.7550    >  -3.8004
    
LinearModel: 5.6K
    Testing:       -2.4744  -3.1292  -2.2824  -2.0288    >  -2.4787
    
NaiveModel: 0
    Testing:       0.9190   0.9190   0.9190   0.9190     >  0.9190
    

--------------------------------------------------------------------------------------------------------------



--------------------------------------------------------------------------------------------------------------

STD

--------------------------------------------------------------------------------------------------------------

                    o_std    h_std    l_std    c_std    
StockGPT: 3.2M
    Testing:       0.0078   0.00

|██████████| 100.0% (00:33) Evaluating model on testing data... (387/388) [1164/1164]: 